# FastVLM-7B server for the Reachy Mini voice agent

Runs Apple's **FastVLM-7B** (highest config) on a Colab GPU and exposes a `POST /query` endpoint that the Reachy voice agent's `FastVLMBackend` can call.

## How to use

1. Runtime → Change runtime type → **GPU** (T4 / L4 / A100; A100 strongly recommended for 7B).
2. Run all cells top to bottom.
3. The last cell prints a public `https://<random>.trycloudflare.com` URL.
4. On the machine running the agent, set:
   ```bash
   export VISION_BACKEND=fastvlm
   export FASTVLM_URL=https://<random>.trycloudflare.com
   ```
5. Launch the agent (`python -m reachy_voice`). The `look_and_describe` tool now hits this server.

## VRAM footprint
FastVLM-7B in fp16 ≈ **14 GB** VRAM. T4 (16 GB) just fits. L4 / A100 / H100 are comfortable. If you OOM on T4, set `LOAD_4BIT = True` in the loader cell.

Keep the Colab tab open while you use the agent — closing it tears down the tunnel and the model.

## 1. Install FastVLM and dependencies

In [ ]:
# Apple's FastVLM is a LLaVA-style fork. We clone it and pip install
# from source — the model code (custom vision tower FastViTHD,
# Qwen2-7B LLM head) is not on PyPI.
!git clone https://github.com/apple/ml-fastvlm.git /content/ml-fastvlm
%cd /content/ml-fastvlm
!pip install -q -e .
!pip install -q fastapi uvicorn nest-asyncio pillow huggingface_hub accelerate

## 2. Download the FastVLM-7B checkpoint

Pulls weights from Hugging Face (~14 GB). Cached in Colab's local disk for the session.

In [ ]:
from huggingface_hub import snapshot_download

MODEL_REPO = "apple/FastVLM-7B"
MODEL_DIR = "/content/checkpoints/FastVLM-7B"

snapshot_download(repo_id=MODEL_REPO, local_dir=MODEL_DIR, local_dir_use_symlinks=False)
!ls -lh {MODEL_DIR}

## 3. Load model + define inference function

Uses Apple's LLaVA-style `load_pretrained_model` from `ml-fastvlm`. If the loader signature differs in your fork (Apple sometimes ships variants), adjust accordingly — the rest of the notebook just needs `predict(pil_image, prompt) -> str`.

In [ ]:
import torch
from PIL import Image

LOAD_4BIT = False  # flip to True on a 16GB GPU if you OOM in fp16

from llava.model.builder import load_pretrained_model
from llava.mm_utils import process_images, tokenizer_image_token, get_model_name_from_path
from llava.conversation import conv_templates
from llava.constants import (
    IMAGE_TOKEN_INDEX,
    DEFAULT_IMAGE_TOKEN,
    DEFAULT_IM_START_TOKEN,
    DEFAULT_IM_END_TOKEN,
)

model_name = get_model_name_from_path(MODEL_DIR)
tokenizer, model, image_processor, context_len = load_pretrained_model(
    model_path=MODEL_DIR,
    model_base=None,
    model_name=model_name,
    load_4bit=LOAD_4BIT,
    device="cuda",
)
model.eval()

# Pick the conversation template matching the LLM backbone (Qwen2 for 7B).
# Apple's repo registers a 'qwen_2' template; fall back to 'llava_v1' if not.
CONV_TEMPLATE = "qwen_2" if "qwen_2" in conv_templates else "llava_v1"
print(f"Using conv template: {CONV_TEMPLATE}")

@torch.inference_mode()
def predict(image: Image.Image, prompt: str, max_new_tokens: int = 128) -> str:
    image = image.convert("RGB")
    image_tensor = process_images([image], image_processor, model.config).to(
        model.device, dtype=torch.float16
    )

    # Build the multimodal prompt: <image> token + user question.
    if model.config.mm_use_im_start_end:
        qs = (
            DEFAULT_IM_START_TOKEN
            + DEFAULT_IMAGE_TOKEN
            + DEFAULT_IM_END_TOKEN
            + "\n"
            + prompt
        )
    else:
        qs = DEFAULT_IMAGE_TOKEN + "\n" + prompt

    conv = conv_templates[CONV_TEMPLATE].copy()
    conv.append_message(conv.roles[0], qs)
    conv.append_message(conv.roles[1], None)
    full_prompt = conv.get_prompt()

    input_ids = (
        tokenizer_image_token(full_prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors="pt")
        .unsqueeze(0)
        .to(model.device)
    )

    output_ids = model.generate(
        input_ids,
        images=image_tensor,
        image_sizes=[image.size],
        do_sample=False,
        temperature=0.0,
        max_new_tokens=max_new_tokens,
        use_cache=True,
    )
    return tokenizer.batch_decode(
        output_ids[:, input_ids.shape[1] :], skip_special_tokens=True
    )[0].strip()

# Quick sanity check on a tiny dummy image.
_dummy = Image.new("RGB", (336, 336), color="red")
print("sanity:", predict(_dummy, "What color is this image?")[:200])

## 4. FastAPI server with `/query` endpoint

Same JSON contract as `reachy_voice.vision.fastvlm.FastVLMBackend`:

```
POST /query
{ "image_b64": "<jpeg base64>", "question": "..." }
→ { "answer": "...", "latency_ms": <float> }
```

In [ ]:
import base64
import io
import time
import threading

import nest_asyncio
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel

nest_asyncio.apply()
app = FastAPI(title="FastVLM-7B")

class QueryReq(BaseModel):
    image_b64: str
    question: str
    max_new_tokens: int = 128

@app.get("/healthz")
def healthz():
    return {"ok": True, "model": MODEL_REPO}

@app.post("/query")
def query(req: QueryReq):
    img = Image.open(io.BytesIO(base64.b64decode(req.image_b64)))
    t0 = time.monotonic()
    answer = predict(img, req.question, max_new_tokens=req.max_new_tokens)
    return {"answer": answer, "latency_ms": (time.monotonic() - t0) * 1000.0}

def _run():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

_server_thread = threading.Thread(target=_run, daemon=True)
_server_thread.start()
time.sleep(2)
print("FastAPI listening on :8000")

## 5. Public tunnel via Cloudflare

Zero signup, no auth token. Cloudflare's quick tunnel mints a `https://<random>.trycloudflare.com` URL pointing at our local FastAPI server. Copy this URL into `FASTVLM_URL` on the agent host.

In [ ]:
import re
import subprocess
import time

!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

public_url = None
deadline = time.time() + 60
while time.time() < deadline:
    line = _proc.stdout.readline()
    if not line:
        time.sleep(0.2)
        continue
    print(line.rstrip())
    m = re.search(r"https://[\w.-]+\.trycloudflare\.com", line)
    if m:
        public_url = m.group(0)
        break

if not public_url:
    raise RuntimeError("cloudflared did not print a tunnel URL within 60s — re-run the cell.")

print("\n" + "=" * 60)
print(f"FASTVLM_URL={public_url}")
print("=" * 60)

## 6. Smoke test from inside Colab

In [ ]:
import base64
import io
import requests
from PIL import Image

img = Image.new("RGB", (336, 336), color="blue")
buf = io.BytesIO()
img.save(buf, format="JPEG")
b64 = base64.b64encode(buf.getvalue()).decode("ascii")

r = requests.post(
    f"{public_url}/query",
    json={"image_b64": b64, "question": "What color is this image?"},
    timeout=60,
)
print(r.status_code, r.json())

## 7. (Optional) Keep the Colab session alive

Colab tears idle GPU sessions down after ~90 min. Run this in a background cell to ping the model and prevent shutdown while you talk to Reachy.

In [ ]:
import time
import requests

while True:
    try:
        requests.get(f"{public_url}/healthz", timeout=5)
    except Exception as e:
        print("keepalive ping failed:", e)
    time.sleep(300)